# 02 — Activation Analysis

**Amaç:** Forward hooks kullanarak iç aktivasyonları gözlemlemek. Bu aşama korelasyon/aday feature üretir; tek başına nedensellik kanıtı değildir. Zero ratio için ReLU çıktıları gözlenir.

In [ ]:
import os, sys
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path: sys.path.append(ROOT)
from src.model import build_model
from src.hooks import register_activation_hooks, remove_hooks
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model(42).to(device)
model.load_state_dict(torch.load('../results/baseline_model.pt', map_location=device))
model.eval()
transform = transforms.ToTensor()
test = datasets.MNIST('data', train=False, download=True, transform=transform)
loader = DataLoader(test, batch_size=256, shuffle=False)
activations, handles = register_activation_hooks(model, ['net.2', 'net.4'])
class_sums = {name: torch.zeros(10, model.net[int(name.split('.')[1])].inplace if False else 1) for name in []}
all_acts = {'net.2': [], 'net.4': []}
all_labels = []
with torch.no_grad():
    for x, y in loader:
        _ = model(x.to(device))
        for name in all_acts: all_acts[name].append(activations[name].cpu())
        all_labels.append(y)
for name in all_acts:
    A = torch.cat(all_acts[name], dim=0)
    Y = torch.cat(all_labels, dim=0)
    print(name, 'shape=', tuple(A.shape), 'mean=', float(A.mean()), 'max=', float(A.max()), 'zero_ratio=', float((A == 0).float().mean()))
    class_means = torch.stack([A[Y == c].mean(dim=0) for c in range(10)])
    candidate = int(class_means.var(dim=0).argmax())
    print('candidate_neuron_by_class_variance:', candidate)
remove_hooks(handles)

## Çıktılar
- Ortalama aktivasyon
- Maksimum aktivasyon
- Zero ratio
- Sınıf bazlı aktivasyon davranışı
- Aday feature/neuron

Aday seçim sonucu yalnızca hipotezdir. Nedensel rol ablation/intervention ile test edilmelidir.